# Track and version prompts with Weave and W&B Registry

In this notebook you will build a small calendar assistant with [Pydantic AI](https://pydantic.dev/docs/ai/overview/), then use Weave and W&B Registry to track the prompt behind that assistant.

The assistant uses a local tool, `find_available_slots`, to search sample calendar availability. That keeps the agent simple enough to inspect while still showing a realistic prompt workflow: load a prompt from disk, publish it to Weave, link it to [Registry](https://docs.wandb.ai/models/registry), and trace how that prompt behaves in a multi-turn conversation.

## Import dependencies

Start by importing the libraries and local project files used throughout the notebook. The standard library imports handle paths, JSON metadata, hidden credential entry, and type annotations. The project import, `find_available_slots`, gives the assistant a tool it can call. Weave records prompts and traces, W&B logs versioned artifacts, and Pydantic AI runs the agent.

In [ ]:
import os
import json
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path
from tools.availability import find_available_slots
from typing import Any, Sequence

import weave
import wandb
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessage

# Reolve the base directory of the project
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name != "calender_assistant":
    BASE_DIR = BASE_DIR / "calender_assistant"

DEFAULT_PROMPT_DIR = BASE_DIR / "prompts"    

## Prequisites

### Add your OpenAI API key
Run the next cell and paste your OpenAI API key when prompted. The key is stored in the notebook process as an environment variable, so you do not need to write it into the notebook.

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI key: ")

## Configure your W&B project

Set the W&B entity and project where this notebook should log artifacts and Weave traces.

Use your own entity name if you are running this outside the shared demo account. The `WEAVE_PROJECT` value uses the same entity and project so the prompt version, Registry link, and conversation traces stay connected.

In [ ]:
ENTITY = "wandb"
PROJECT = "pydanticai_demo"
WEAVE_PROJECT = f"{ENTITY}/{PROJECT}"

## Define the assistant helpers

These helper functions keep the rest of the notebook focused on the prompt workflow.

`CalendarAgent` stores the agent, model, agent name, and prompt text together so we can pass one object through the demo. `build_agent()` creates the Pydantic AI agent and registers the local availability tool. `run_turn_async()` runs one user turn inside a Weave trace. `load_prompt_info()` reads both the prompt text and its metadata from the `prompts/` directory.

In [ ]:
@dataclass
class CalendarAgent:
    agent: Agent
    model: str
    agent_name: str
    prompt: str


def build_agent(model: str, agent_name: str, prompt: str) -> CalendarAgent:
    """Build the calendar assistant and register its local tools."""
    calendar_agent = Agent(
        model=model,
        output_type=str,
        name=agent_name,
        instructions=prompt,
    )
    calendar_agent.tool_plain(find_available_slots)
    return CalendarAgent(
        agent=calendar_agent,
        model=model,
        agent_name=agent_name,
        prompt=prompt,
    )


async def run_turn_async(
    calendar_agent: CalendarAgent,
    user_input: str,
    message_history: Sequence[ModelMessage] | None = None,
) -> list[ModelMessage]:
    print(f"\nUser: {user_input}")

    with weave.start_turn(
        user_message=user_input,
        model=calendar_agent.model,
        agent_name=calendar_agent.agent_name,
        system_instructions=[calendar_agent.prompt],
    ):
        result = await calendar_agent.agent.run(
            user_input,
            message_history=message_history,
        )

    print(f"Assistant: {result.output}")
    return list(result.all_messages())


def load_prompt_info(prompt_dir: Path) -> tuple[str, dict[str, Any]]:
    """Load the prompt text and manifest metadata from the prompt directory."""
    with (prompt_dir / "manifest.json").open(encoding="utf-8") as manifest_file:
        manifest = json.load(manifest_file)

    # Read the prompt text from the file specified in the manifest
    prompt_file = prompt_dir / manifest["prompt_file"]
    prompt = prompt_file.read_text(encoding="utf-8")

    return prompt, manifest

## Log the code as a W&B Artifact

Before publishing the prompt, log the code that produced it. This step creates a versioned code artifact that includes the local tool, this notebook, the Python script version of the demo, and the requirements file.

That artifact gives the prompt useful lineage: later, when you inspect a prompt version or trace, you can see which code version was used to run it.

In [ ]:

with wandb.init(entity=ENTITY, project=PROJECT, job_type="publish-code") as run:
    code_artifact = wandb.Artifact(name="calendar-assistant-code", type="code")
    code_artifact.add_dir(str(BASE_DIR / "tools"), name="tools")
    code_artifact.add_file(str(BASE_DIR / "calendar_assist_weave_agents_registry.py"))
    code_artifact.add_file(str(BASE_DIR / "calendar_assistant.ipynb"))
    code_artifact.add_file(str(BASE_DIR / "requirements.txt"))
    logged_code_artifact = run.log_artifact(code_artifact)
    logged_code_artifact.wait() # Wait for the artifact to finish logging before proceeding

    # Store the artifact reference in the manifest for tracking purposes
    code_artifact_ref = logged_code_artifact.qualified_name

## Publish the prompt to Weave and link it to Registry

Publish the prompt as a Weave `StringPrompt`, then link that published prompt version to a W&B Registry collection.

The following code snippet converts our `prompt.md` file (defined in `prompts/prompt.md`) into a Weave-friendly object `weave.StringPrompt`. We then publish the 

1. Initialize a Weave client with `weave.init()`.
2. Create a Weave prompt object with a built-in prompt class `weave.StringPrompt()`.
3. Publish the prompt to Weave with `weave.publish()`
4. Link the published prompt version to a collection in a registry with `weave_client.link_prompt_to_registry()`. Pass the destination collection path to the target_path parameter.

The path consists of the `wandb-registry-` prefix, the registry name, and the collection name:

```text
wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}
```

In [ ]:
# Load the prompt and manifest metadata from the specified prompt directory
prompt, manifest = load_prompt_info(DEFAULT_PROMPT_DIR)

model = manifest.get("model")
agent_name = manifest.get("agent_name")
registry_target = f"wandb-registry-{manifest.get('registry')}/{manifest.get('collection')}"

In [ ]:
# Initialize Weave, publish the prompt to Weave, and link it to the W&B Registry
weave_client = weave.init(WEAVE_PROJECT)

weave_prompt_ref = weave.publish(
    weave.StringPrompt(prompt),
    name=manifest["name"]
)
weave_client.link_prompt_to_registry(
    prompt=weave_prompt_ref,
    target_path=registry_target
)

## Run a traced conversation

Finally, build the calendar assistant with the prompt you just published and run a short two-turn conversation.

The `weave.start_conversation()` block groups the turns into one conversation trace. Each `weave.start_turn()` call records the user message, model, agent name, and system instructions for that turn. The trace also includes metadata that points back to the prompt version, Registry target, and code artifact, which gives you a full path from behavior back to the exact prompt and code used to produce it.

In [ ]:
# Build the calendar assistant agent with the loaded prompt and model
calendar_agent = build_agent(model, agent_name, prompt)
message_history: list[ModelMessage] | None = None

with weave.start_conversation(
    agent_name=calendar_agent.agent_name,
    model=calendar_agent.model,
    conversation_name="calendar-assistant-demo",
    attributes={
        "prompt.name": manifest.get("name", ""),
        "prompt.artifact_type": manifest.get("artifact_type", ""),
        "prompt.registry_target": registry_target,
        "prompt.weave_ref": weave_prompt_ref.uri,
        "code.artifact_ref": code_artifact_ref,
    },
):
    message_history = await run_turn_async(
        calendar_agent,
        "I want to schedule a meeting next week. Can you help me find available time slots?",
        message_history,
    )

    message_history = await run_turn_async(
        calendar_agent,
        "Wednesday afternoon for 30 minutes.",
        message_history,
    )